In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/bench/checkpoints/pre_cell_4.pickle

In [4]:
%%RecordEvent
%%time
### cell 4 ###

def create_dataframe_of_counts_16(dataframe, column, rename_index, rename_column, return_percentages=False):
    # drop the first row using GPU-based iloc (no CPU __getitem__ here)
    df = dataframe.iloc[1:]
    # compute raw counts on GPU via groupby.size()
    vc = df.groupby(column).size()
    if return_percentages:
        # convert to percentages on GPU
        vc = vc / len(df) * 100
    # reset_index and rename entirely on GPU
    df_counts = vc.reset_index()
    return df_counts.rename(columns={
        column:     rename_index,
        'size':     rename_column
    })

# call the optimized function (no slicing outside)
percentages_per_country_df = create_dataframe_of_counts_16(
    responses_df_2022_cell10,
    'In which country do you currently reside?',
    'country',
    '% of respondents',
    return_percentages=False
)

# inspect the resulting GPU DataFrame
percentages_per_country_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   country  58 non-null     object
 1   0        58 non-null     int64 
dtypes: int64(1), object(1)
memory usage: 1.0+ KB
CPU times: user 130 ms, sys: 0 ns, total: 130 ms
Wall time: 130 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small/checkpoints/post_cell_4_try_1.pickle